# Colab → Deepnote → Colab via Greft

This notebook runs a small Iris classification experiment, sends its metrics to a Deepnote reviewer through Greft, and then reads the review returned by Deepnote.

```text
Colab @colab-agent
    ↓ experiment_result
Greft
    ↓
Deepnote @deepnote-agent
    ↓ experiment_review
Greft
    ↓
Colab
```


## 1. Install dependencies

In [1]:
%pip install -q "mcp" "httpx2" "scikit-learn"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.9 MB/s eta 0:00:00


## 2. Load Colab secrets

Add `GREFT_API_URL` and `GREFT_API_KEY` in the Colab **Secrets** panel before running this cell.


In [6]:
from google.colab import userdata
import json
import uuid
import httpx2

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

GREFT_API_URL = userdata.get("GREFT_API_URL").rstrip("/")
GREFT_API_KEY = userdata.get("GREFT_API_KEY")

COLAB_ADDRESS = "@ralph-planner"
DEEPNOTE_ADDRESS = "@reviewer"

print("Colab configuration loaded.")


Colab configuration loaded.


## 3. Greft MCP helper

Each tool call opens a fresh authenticated Streamable HTTP MCP session. The inbox helper ignores unrelated plain-text or email messages and looks only for JSON messages with the expected `kind`.


In [7]:
async def call_greft_tool(address, tool_name, arguments=None):
    mcp_url = f"{GREFT_API_URL}/mcp?address={address}"

    async with httpx2.AsyncClient(
        headers={"Authorization": f"Bearer {GREFT_API_KEY}"},
        timeout=httpx2.Timeout(30.0, read=300.0),
    ) as http_client:
        async with streamable_http_client(
            mcp_url,
            http_client=http_client,
        ) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                return await session.call_tool(
                    tool_name,
                    arguments or {},
                )


def get_payload_text(message):
    payload = message.get("payload") or {}

    return (
        payload.get("text")
        or payload.get("message")
        or payload.get("body")
    )


def find_latest_json_message(messages, kind, experiment_id=None):
    ordered = sorted(
        messages,
        key=lambda message: message.get("seq", -1),
        reverse=True,
    )

    for message in ordered:
        text = get_payload_text(message)

        if not isinstance(text, str):
            continue

        try:
            data = json.loads(text)
        except json.JSONDecodeError:
            continue

        if data.get("kind") != kind:
            continue

        if (
            experiment_id is not None
            and data.get("experiment_id") != experiment_id
        ):
            continue

        return message, data

    return None, None


## 4. Run the experiment

This example uses scikit-learn's built-in Iris dataset and Logistic Regression. A unique experiment ID prevents a review from an older run being mistaken for the current one.


In [8]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

model = LogisticRegression(max_iter=300)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

EXPERIMENT_ID = f"iris-logreg-{uuid.uuid4().hex[:8]}"

experiment = {
    "kind": "experiment_result",
    "experiment_id": EXPERIMENT_ID,
    "dataset": "Iris",
    "model": "LogisticRegression",
    "accuracy": round(float(accuracy), 4),
    "training_rows": len(X_train),
    "test_rows": len(X_test),
}

print(json.dumps(experiment, indent=2))


{
  "kind": "experiment_result",
  "experiment_id": "iris-logreg-b4452505",
  "dataset": "Iris",
  "model": "LogisticRegression",
  "accuracy": 0.9667,
  "training_rows": 120,
  "test_rows": 30
}


## 5. Send the experiment to Deepnote

After running this cell, switch to `deepnote.ipynb`.


In [9]:
await call_greft_tool(
    COLAB_ADDRESS,
    "send_message",
    {
        "to": DEEPNOTE_ADDRESS,
        "message": json.dumps(experiment),
    },
)

print(f"Sent {EXPERIMENT_ID} to {DEEPNOTE_ADDRESS}.")


Sent iris-logreg-b4452505 to @reviewer.


## 6. Read Deepnote's review

Run this cell **after Deepnote has sent its review**. `read_messages` is used instead of `wait_for_message` so older unacknowledged mailbox messages do not interfere with the workflow.


In [10]:
result = await call_greft_tool(
    COLAB_ADDRESS,
    "read_messages",
    {},
)

messages = (result.structured_content or {}).get("messages", [])

review_message, review = find_latest_json_message(
    messages,
    kind="experiment_review",
    experiment_id=EXPERIMENT_ID,
)

if review is None:
    raise RuntimeError(
        f"No review found yet for {EXPERIMENT_ID}. "
        "Run the Deepnote review cells first, then try again."
    )

print("Experiment review")
print("-----------------")
print("Experiment:", review["experiment_id"])
print("Verdict:   ", review["verdict"])
print("Accuracy:  ", review["accuracy"])
print("Required:  ", review["required_accuracy"])


Experiment review
-----------------
Experiment: iris-logreg-b4452505
Verdict:    PASS
Accuracy:   0.9667
Required:   0.9


## Expected result

The final output should look similar to:

```text
Experiment review
-----------------
Experiment: iris-logreg-...
Verdict:    PASS
Accuracy:   0.9667
Required:   0.9
```

Before committing this notebook, clear any output containing real account-specific IDs or messages.
